# Transformer Block: Residuals and LayerNorm

AI & ML Foundations: Attention and Transformers · *Mathematics for AI and Machine Learning* by Fuheng Wu

Stack blocks with and without residual connections and LayerNorm and track how the signal and its gradient survive depth.

Companion code for the [interactive demo](https://math4ai.org/companion/ml/transformer-block/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(21)

# The demo's defaults: 32 stacked blocks of width 32; toggle the residual and LayerNorm
D, depth, T = 32, 32, 8


class Block(torch.nn.Module):
    """Pre-LN Transformer block: x + Attn(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, residual, layernorm):
        super().__init__()
        self.res, self.ln = residual, layernorm
        self.n1, self.n2 = torch.nn.LayerNorm(D), torch.nn.LayerNorm(D)
        self.attn = torch.nn.MultiheadAttention(D, 4, batch_first=True)
        self.mlp = torch.nn.Sequential(torch.nn.Linear(D, 4 * D), torch.nn.GELU(), torch.nn.Linear(4 * D, D))

    def forward(self, x):
        u = self.n1(x) if self.ln else x
        a = self.attn(u, u, u, need_weights=False)[0]
        x = x + a if self.res else a
        u = self.n2(x) if self.ln else x
        m = self.mlp(u)
        return x + m if self.res else m


print(" residual  LayerNorm   |x| at the end   |dL/dx0|   (input-gradient size)")
for res, ln in ((True, True), (True, False), (False, True), (False, False)):
    net = torch.nn.Sequential(*[Block(res, ln) for _ in range(depth)]).double()   # float64: tiny gradients stay visible
    x0 = torch.randn(1, T, D, dtype=torch.float64, requires_grad=True)
    out = net(x0)
    out.pow(2).mean().backward()
    print(f"   {str(res):5s}     {str(ln):5s}       {out.norm().item():10.3e}     {x0.grad.norm().item():.3e}")
# Without the residual path the gradient must pass through every sublayer and fades;
# the identity path x + F(x) gives it a direct route back to the input.

 residual  LayerNorm   |x| at the end   |dL/dx0|   (input-gradient size)
   True      True         4.558e+01     1.248e+00
   True      False        2.361e+02     2.872e+02
   False     True         2.557e+00     1.928e-04
   False     False        9.374e-01     7.101e-40
